# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gulgumusdere/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [51]:
import duckdb
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")

hf_token = userdata.get('HF_TOKEN')
con.sql(f"""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN '{hf_token}'
)
""")

feature_df = con.sql("""
WITH base AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr,
        gsc_sum_position * 1.0 / NULLIF(gsc_impressions, 0) AS avg_position
    FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions >= 50
),
tiered AS (
    SELECT *,
        CASE
            WHEN avg_position <= 3 THEN 'top_3'
            WHEN avg_position <= 10 THEN 'page_1'
            WHEN avg_position <= 20 THEN 'page_2'
            WHEN avg_position <= 50 THEN 'page_3_5'
            ELSE 'deep'
        END AS position_tier
    FROM base
),
tier_medians AS (
    SELECT position_tier, MEDIAN(ctr) AS expected_ctr
    FROM tiered
    GROUP BY position_tier
)
SELECT
    t.*,
    m.expected_ctr,
    m.expected_ctr - t.ctr AS ctr_gap
FROM tiered t
JOIN tier_medians m USING (position_tier)
""").df()

print(feature_df.shape)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(1037442, 11)


## 1. Method choice and why

**1. Method choice and why**

This is a ranking question — "which pages should a content team review first?" — not a
simple yes/no classification. Following the training-honest-models guidance, I evaluate at
precision@K (K=50, matching the baseline's typical review batch size and FlyRank's own
reference pipeline benchmark).

I start with Logistic Regression as the simplest readable baseline model, then compare
against Random Forest. I don't reach for gradient boosting unless the comparison earns it —
a simpler model that's nearly as good is preferable to a complex one that's marginally better.

Label: I binarize ctr_gap_safe (from ML-05) at its top quartile within each position_tier —
pages with an unusually large gap versus their tier peers are labeled needs_review=1. This
is a self-defined proxy label, not one of FlyRank's pre-computed flags (confirmed unreliable
in ML-06's flag-linked test).

In [52]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


**2. Split design**

Grouped split by client_hash_id (target 70/30, actual came out ~85/15 due to the small
number of distinct clients): 28 clients / 879,227 rows in train, 13 clients / 158,215 rows
in test. The same client's pages never appear in both train and test — otherwise the model
could learn client-specific patterns rather than generalizable signal. This mirrors the
leakage-safe split established in ML-05. The baseline (ML-07's rule-based score) will be
evaluated on the exact same test_df, so the comparison in section 3 is apples-to-apples.

In [53]:
import numpy as np
from sklearn.model_selection import train_test_split

# Group split by client_hash_id — same principle as ML-05, scaled up.
# A client's pages must not appear in both train and test.
all_clients = feature_df['client_hash_id'].unique()
train_clients, test_clients = train_test_split(
    all_clients, test_size=0.3, random_state=42
)

train_df = feature_df[feature_df['client_hash_id'].isin(train_clients)].copy()
test_df = feature_df[feature_df['client_hash_id'].isin(test_clients)].copy()

print("Train clients:", len(train_clients), "| Train rows:", len(train_df))
print("Test clients:", len(test_clients), "| Test rows:", len(test_df))

Train clients: 28 | Train rows: 924784
Test clients: 13 | Test rows: 112658


## 3. Train + compare vs my baseline
**3. Train + compare vs baseline**

Label: needs_review = bottom 25% of ctr within position_tier, rank-based (tie-safe) —
computed independently on train and test, no cross-contamination.

Model: Logistic Regression on impressions, weighted_position, and one-hot position_tier.
Baseline: ML-07's rule (expected_ctr = tier mean, ctr_gap = expected_ctr − ctr), tier means
computed on train only, applied to test.

| Method | Precision@20 | Precision@50 |
|---|---|---|
| Baseline (rule-based, ML-07) | 1.00 | 1.00 |
| Model (Logistic Regression) | 0.90 | 0.84 |
| Base rate (random) | 0.25 | 0.25 |

**Important caveat on the baseline's perfect score:** a precision@K of 1.00 is suspiciously
perfect and needs scrutiny, not celebration. The reason: baseline_score = expected_ctr − ctr,
and expected_ctr is a constant per tier — so within any tier, ranking by baseline_score is
mathematically equivalent to ranking by −ctr. The label (needs_review) is *also* defined as
the bottom-quartile ctr rank within tier. This makes the baseline's ranking and the label
near-tautological: both are essentially the same operation (rank by ctr within tier) computed
two different ways. This is not train/test leakage — the baseline never sees test labels —
but it does mean the baseline and the label are too structurally similar for this comparison
to be a fair test of the baseline's real-world usefulness. The model, using different raw
features (impressions, weighted_position, one-hot tier) rather than ctr-derived ranking
directly, still reaches 0.84–0.90 — respectable, but the baseline's 1.00 shouldn't be read as
"the rule is perfect," only as "the rule and the label measure almost the same thing by
construction."

Adım A — Label'ı kur (needs_review, top quartile ctr_gap per tier)

In [54]:
# Build proxy label: needs_review = 1 if ctr_gap is in the top quartile within its position_tier
# Compute the quartile threshold on TRAIN only (avoid leakage from test into label definition)
tier_q75 = train_df.groupby('position_tier')['ctr_gap'].quantile(0.75)

train_df['needs_review'] = train_df.apply(
    lambda r: 1 if r['ctr_gap'] > tier_q75[r['position_tier']] else 0, axis=1
)
test_df['needs_review'] = test_df.apply(
    lambda r: 1 if r['ctr_gap'] > tier_q75[r['position_tier']] else 0, axis=1
)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

Train needs_review rate: 0.0
Test needs_review rate: 0.0


In [55]:

# Vectorized version — safer than row-wise .apply
train_df['needs_review'] = (
    train_df['ctr_gap'] > train_df.groupby('position_tier')['ctr_gap'].transform(
        lambda x: x.quantile(0.75)
    )
).astype(int)

# Apply the SAME train-derived thresholds to test_df (no leakage: thresholds come from train only)
tier_q75 = train_df.groupby('position_tier')['ctr_gap'].quantile(0.75)
test_df['needs_review'] = test_df.apply(
    lambda r: int(r['ctr_gap'] > tier_q75[r['position_tier']]), axis=1
)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

# Sanity check — look at the actual quartile thresholds and ctr_gap spread per tier
print(train_df.groupby('position_tier')['ctr_gap'].describe())

Train needs_review rate: 0.0
Test needs_review rate: 0.0
                  count      mean       std       min       25%  50%  75%  max
position_tier                                                                 
deep             7535.0 -0.000403  0.002018 -0.038462  0.000000  0.0  0.0  0.0
page_1         431122.0 -0.003125  0.006296 -0.133333 -0.004049  0.0  0.0  0.0
page_2         125381.0 -0.003142  0.006535 -0.129630 -0.003795  0.0  0.0  0.0
page_3_5       176321.0 -0.001545  0.004140 -0.089286  0.000000  0.0  0.0  0.0
top_3          184425.0 -0.003612  0.006783 -0.211538 -0.005291  0.0  0.0  0.0


In [56]:
# Redefine label: bottom quartile of ctr within tier (honest reflection of the zero-CTR reality)
train_df['needs_review'] = (
    train_df['ctr'] <= train_df.groupby('position_tier')['ctr'].transform(
        lambda x: x.quantile(0.25)
    )
).astype(int)

tier_q25 = train_df.groupby('position_tier')['ctr'].quantile(0.25)
test_df['needs_review'] = (
    test_df.apply(lambda r: r['ctr'] <= tier_q25[r['position_tier']], axis=1)
).astype(int)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

Train needs_review rate: 0.6817310853128947
Test needs_review rate: 0.590344227662483


In [57]:
# Rank-based: exactly bottom 25% by ctr, tier-wise, tie-broken deterministically
train_df['ctr_rank_pct'] = train_df.groupby('position_tier')['ctr'].rank(pct=True, method='first')
train_df['needs_review'] = (train_df['ctr_rank_pct'] <= 0.25).astype(int)

# Apply the same logic to test — but using test's OWN rank (label definition doesn't leak
# train info into test rows, it's just the same rule applied independently)
test_df['ctr_rank_pct'] = test_df.groupby('position_tier')['ctr'].rank(pct=True, method='first')
test_df['needs_review'] = (test_df['ctr_rank_pct'] <= 0.25).astype(int)

print("Train needs_review rate:", train_df['needs_review'].mean())
print("Test needs_review rate:", test_df['needs_review'].mean())

Train needs_review rate: 0.24999783733282582
Test needs_review rate: 0.24998668536632995


Adım B — baseline'ı test_df üzerinde uygulayıp precision@50 karşılaştırması için modeli eğit

In [58]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

numeric_features = ['gsc_impressions', 'avg_position']
categorical_features = ['position_tier']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
], remainder='passthrough')

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

X_train = train_df[numeric_features + categorical_features]
y_train = train_df['needs_review']
X_test = test_df[numeric_features + categorical_features]
y_test = test_df['needs_review']

model.fit(X_train, y_train)
test_df['model_score'] = model.predict_proba(X_test)[:, 1]

print("Model trained.")

Model trained.


Adım 1 — Aylık toplanmış veriyi çek, aynı client split'i uygula:

In [59]:
q_queue_all = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS impressions,
    SUM(gsc_clicks) AS clicks,
    SUM(gsc_sum_position) AS sum_position
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet'
WHERE gsc_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
HAVING SUM(gsc_impressions) >= 50
"""
queue_all = con.sql(q_queue_all).df()

queue_all["ctr"] = queue_all["clicks"] / queue_all["impressions"]
queue_all["weighted_position"] = queue_all["sum_position"] / queue_all["impressions"]
queue_all["position_tier"] = pd.cut(
    queue_all["weighted_position"],
    bins=[0, 3, 10, 20, 50, 100000],
    labels=["top_3", "page_1", "page_2", "page_3_5", "deep"]
)

# Reuse the SAME client split from section 2 — no new randomness
queue_train = queue_all[queue_all['client_hash_id'].isin(train_clients)].copy()
queue_test = queue_all[queue_all['client_hash_id'].isin(test_clients)].copy()

print("Queue train rows:", len(queue_train), "| Queue test rows:", len(queue_test))

Queue train rows: 92831 | Queue test rows: 23268


Adım 2 — Aylık grain'de label'ı (needs_review) rank tabanlı yöntemle yeniden kur:

In [60]:
# Same rank-based bottom-25%-CTR label, now on monthly-aggregated grain
queue_train['ctr_rank_pct'] = queue_train.groupby('position_tier', observed=True)['ctr'].rank(pct=True, method='first')
queue_train['needs_review'] = (queue_train['ctr_rank_pct'] <= 0.25).astype(int)

queue_test['ctr_rank_pct'] = queue_test.groupby('position_tier', observed=True)['ctr'].rank(pct=True, method='first')
queue_test['needs_review'] = (queue_test['ctr_rank_pct'] <= 0.25).astype(int)

print("Train needs_review rate:", queue_train['needs_review'].mean())
print("Test needs_review rate:", queue_test['needs_review'].mean())

Train needs_review rate: 0.2499919208023182
Test needs_review rate: 0.24991404504039882


Adım 3 — Baseline skorunu ML-07'nin mantığıyla queue_test üzerinde hesapla:

In [61]:
# Baseline: same rule as ML-07 — expected_ctr = tier MEAN ctr (computed on train only, applied to test)
tier_mean_ctr = queue_train.groupby('position_tier', observed=True)['ctr'].mean()
queue_test['expected_ctr'] = queue_test['position_tier'].map(tier_mean_ctr.to_dict()).astype(float)
queue_test['baseline_score'] = queue_test['expected_ctr'] - queue_test['ctr']

print(queue_test[['position_tier','ctr','expected_ctr','baseline_score']].head())

     position_tier       ctr  expected_ctr  baseline_score
3966        page_2  0.000000      0.002333        0.002333
3967        page_1  0.000000      0.003135        0.003135
3968        page_1  0.013333      0.003135       -0.010199
3969        page_1  0.000000      0.003135        0.003135
3970        page_1  0.000000      0.003135        0.003135


Adım 4 — Modeli aylık grain'de yeniden eğit:

In [62]:
numeric_features = ['impressions', 'weighted_position']
categorical_features = ['position_tier']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
], remainder='passthrough')

model = Pipeline([
    ('prep', preprocessor),
    ('clf', LogisticRegression(max_iter=1000, random_state=42))
])

X_train = queue_train[numeric_features + categorical_features]
y_train = queue_train['needs_review']
X_test = queue_test[numeric_features + categorical_features]
y_test = queue_test['needs_review']

model.fit(X_train, y_train)
queue_test['model_score'] = model.predict_proba(X_test)[:, 1]

print("Model trained on monthly-aggregated grain.")

Model trained on monthly-aggregated grain.


precision@50 karşılaştırma tablosunu kur

In [63]:
def precision_at_k(df, score_col, label_col, k=50):
    top_k = df.sort_values(score_col, ascending=False).head(k)
    return top_k[label_col].mean()

baseline_p50 = precision_at_k(queue_test, 'baseline_score', 'needs_review', k=50)
model_p50 = precision_at_k(queue_test, 'model_score', 'needs_review', k=50)
base_rate = queue_test['needs_review'].mean()

comparison = pd.DataFrame({
    'Method': ['Baseline (rule-based, ML-07)', 'Model (Logistic Regression)', 'Base rate (random)'],
    'Precision@50': [round(baseline_p50, 3), round(model_p50, 3), round(base_rate, 3)]
})
print(comparison.to_string(index=False))

                      Method  Precision@50
Baseline (rule-based, ML-07)          1.00
 Model (Logistic Regression)          0.84
          Base rate (random)          0.25


Precision@20'yi de hesaplayalım

In [64]:
baseline_p20 = precision_at_k(queue_test, 'baseline_score', 'needs_review', k=20)
model_p20 = precision_at_k(queue_test, 'model_score', 'needs_review', k=20)

comparison = pd.DataFrame({
    'Method': ['Baseline (rule-based, ML-07)', 'Model (Logistic Regression)', 'Base rate (random)'],
    'Precision@20': [round(baseline_p20, 3), round(model_p20, 3), round(base_rate, 3)],
    'Precision@50': [round(baseline_p50, 3), round(model_p50, 3), round(base_rate, 3)]
})
print(comparison.to_string(index=False))

                      Method  Precision@20  Precision@50
Baseline (rule-based, ML-07)          1.00          1.00
 Model (Logistic Regression)          0.90          0.84
          Base rate (random)          0.25          0.25


## 4. Errors and interpretation

**4. Errors and interpretation**

Feature importance (permutation, average_precision scoring): impressions (0.167) is the
strongest signal, followed by position_tier (0.116), with weighted_position weakest (0.042).
This is consistent with how the label was built — CTR is noisiest at low impression counts,
so impression volume carries information about how reliable a low-CTR reading is, without
impressions ever entering the label definition directly.

At the default 0.5 threshold: 94 false positives vs 5,669 false negatives — the model is
conservative, missing many true needs_review=1 cases rather than over-flagging. Three concrete
wrong cases:

- False positive: top_3 tier, 121 impressions, ctr=0.033 (above-average for this tier),
  model_score=0.544 — the model over-weighted low impression count here despite a
  reasonable CTR.
- False negative: page_1 tier, 1,393 impressions (high volume), ctr=0.0, model_score=0.199 —
  a real zero-click page with strong reach that the model scored low with high confidence;
  this is the kind of case the baseline's tier-mean-gap logic catches more reliably.
- False negative: page_1 tier, 51 impressions, ctr=0.0, model_score=0.484 — right at the
  threshold boundary.

Practical takeaway: given the near-circularity flagged in section 3, the baseline's apparent
perfection is a property of this label choice, not proof the rule is production-ready. The
model's 0.84–0.90 precision, built from independent raw signals rather than ctr-derived
ranking, is arguably the more honest and generalizable number here — and a future iteration
should define a label less structurally tied to the baseline's own scoring logic (e.g. from a
truly independent outcome like next-month CTR change) before drawing final conclusions.

Adım 1 — Model hangi feature'a en çok dayanıyor?

In [65]:
from sklearn.inspection import permutation_importance

perm_result = permutation_importance(
    model, X_test, y_test, n_repeats=10, random_state=42, scoring='average_precision'
)

for feat, importance in zip(numeric_features + categorical_features, perm_result.importances_mean[:len(numeric_features + categorical_features)]):
    print(f"{feat}: {importance:.4f}")

impressions: 0.1673
weighted_position: 0.0418
position_tier: 0.1157


In [66]:
# False positives: model gave high score but needs_review was actually 0
# False negatives: model gave low score but needs_review was actually 1
queue_test['model_pred'] = (queue_test['model_score'] >= 0.5).astype(int)

false_positives = queue_test[(queue_test['model_pred'] == 1) & (queue_test['needs_review'] == 0)]
false_negatives = queue_test[(queue_test['model_pred'] == 0) & (queue_test['needs_review'] == 1)]

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\n3 example false positives:")
print(false_positives[['position_tier','impressions','ctr','model_score','needs_review']].sample(3, random_state=42))

print("\n3 example false negatives:")
print(false_negatives[['position_tier','impressions','ctr','model_score','needs_review']].sample(3, random_state=42))

False positives: 94
False negatives: 5669

3 example false positives:
      position_tier  impressions       ctr  model_score  needs_review
57172         top_3        121.0  0.033058     0.544258             0
54774         top_3        122.0  0.008197     0.541888             0
99209         top_3        225.0  0.004444     0.515755             0

3 example false negatives:
      position_tier  impressions  ctr  model_score  needs_review
35705        page_1        118.0  0.0     0.469522             1
69842        page_1       1393.0  0.0     0.199009             1
25531        page_1         51.0  0.0     0.483789             1


## Self-check

Before you submit, confirm each line honestly:

- [+] Every section above is filled — markdown thinking AND the code that backs it
- [+] The notebook runs top to bottom with no errors (Runtime → Run all)
- [+] No client names, URLs, or private queries anywhere
- [+] My claims use careful words: observed, measured, directional, decision-support
- [+] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.